# ĐỒ ÁN MÔN HỌC: CÁC PHƯƠNG PHÁP HỌC MÁY
## Đề tài: Ứng dụng Học máy trong Xây dựng Hệ thống Nhận diện Nông sản Việt Nam

**Thành viên thực hiện:**
- Nguyễn Hoàng Anh Khoa (2212394)
- Phan Thanh Khải (2212386)
- Phan Thái Bảo (2212345)
- Lê Thành Thái (2212456)
- Nguyễn Tôn Nữ Thu Thanh (2212459)

**Mô tả:** Notebook này chứa toàn bộ mã nguồn bao gồm: Thu thập dữ liệu từ Hugging Face, Tiền xử lý, Huấn luyện mô hình YOLOv8, và Thực thi chạy thực nghiệm.

## 1. Cài đặt các thư viện cần thiết

In [ ]:
!pip install -U ultralytics huggingface_hub opencv-python

## 2. Quá trình Thu thập dữ liệu
Sử dụng thư viện `huggingface_hub` để tải tập dữ liệu "100 Crops & Plants Object Detection 25k" từ Hugging Face.

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile
import os

# Tên repository trên Hugging Face
repo_id = "devshaheen/100_crops_plants_object_detection_25k_image_dataset"
filename = "leaflogic object detection.v5i.yolov5pytorch.zip"

# Tải file nén về máy
print("Đang tải dữ liệu, vui lòng chờ...")
file_path = hf_hub_download(repo_id=repo_id, filename=filename, repo_type="dataset")

# Thư mục đích để giải nén
extract_dir = "./dataset_nongsan"
os.makedirs(extract_dir, exist_ok=True)

# Giải nén tập dữ liệu YOLO
print("Đang giải nén tập dữ liệu...")
with zipfile.ZipFile(file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"Hoàn tất! Tập dữ liệu đã được giải nén vào thư mục: {extract_dir}")

## 3. Tiền xử lý Dữ liệu
Tập dữ liệu tải về đã được tác giả format sẵn theo chuẩn YOLOv5/YOLOv8 PyTorch (chứa các thư mục images, labels và file data.yaml). Chúng ta chỉ cần chuẩn bị đường dẫn tới file YAML để bắt đầu huấn luyện.

In [ ]:
import yaml

yaml_path = os.path.join(extract_dir, "data.yaml")
print(f"File cấu trúc dữ liệu: {yaml_path}")

with open(yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)
    
print(f"Số lượng class: {data_config['nc']}")
print(f"Danh sách các lớp: {data_config['names']}")

## 4. Quá trình Huấn luyện Mô hình (Model Training)
Sử dụng YOLOv8s làm cấu trúc cơ sở, tiến hành huấn luyện với các kỹ thuật Augmentation để cải thiện mô hình.

In [ ]:
from ultralytics import YOLO

# Khởi tạo mô hình YOLOv8s (Small)
model = YOLO('yolov8s.pt')

# Bắt đầu quá trình huấn luyện
results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    workers=4,
    optimizer='AdamW',
    lr0=0.001,
    mosaic=1.0,  # Bật Data Augmentation (Mosaic)
    mixup=0.2,   # Bật MixUp
    device=0     # Sử dụng GPU
)

print("Quá trình huấn luyện thành công!")

## 5. Chạy Thực nghiệm và Cải tiến Mô hình (YOLO-World Open Vocabulary)
Thực nghiệm chạy mô hình cơ sở YOLOv8s và ứng dụng biến thể YOLO-World (Zero-shot) cho các loại nông sản không có trong tập dữ liệu huấn luyện.

In [ ]:
import cv2
import matplotlib.pyplot as plt

# 1. Đánh giá mô hình đã huấn luyện
metrics = model.val()
print(f"mAP@50 (YOLOv8s Baseline): {metrics.box.map50:.3f}")

# 2. Thử nghiệm mở rộng không gian nhận dạng với YOLO-World
world_model = YOLO('yolov8s-world.pt')

# Thêm các lớp nông sản đặc thù chưa được học
custom_classes = data_config['names'] + ["Avocado", "Dragon fruit", "Rambutan", "Mangosteen"]
world_model.set_classes(custom_classes)

# Dự đoán trên ảnh thực tế
test_image_path = "test_fruit.jpg"  # Thay đổi thành đường dẫn file ảnh của bạn
try:
    results_world = world_model.predict(test_image_path, conf=0.25, iou=0.45)
    res_plotted = results_world[0].plot()
    plt.figure(figsize=(10, 10))
    plt.imshow(cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title("Kết quả nhận dạng với mô hình YOLO-World")
    plt.show()
except Exception as e:
    print(f"Vui lòng chuẩn bị một ảnh 'test_fruit.jpg' để kiểm thử.")
